# Data Preparation

This notebook prepares the data for our SFT experiments using the
Tulu 3 decontaminated dataset.

Run the cells in order from the project root directory.

The pipeline consists of three steps:
1. Download and clean the dataset.
2. Create a fixed validation set.
3. Build training mixtures with different task proportions.

## 1. Download and Clean the Data

Download the Tulu 3 dataset and retain samples relevant to our
three evaluation tasks: Instruction Following, Math, and Coding.

The cleaned data will be saved to `data/full/`.

This step only needs to be run once.

In [ ]:
!python prepare_data.py

## 2. Create the Validation Set

Create a fixed validation set with 600 samples, equally distributed
across Instruction Following, Math, and Coding.

All training mixtures will automatically exclude this validation set.

**Important:** Keep the same validation set for all experiments.
If you replace it, previously generated training data and trained
checkpoints may overlap with the new validation set.

In [ ]:
!python build_mixture.py --create-validation --size 600 --ratios 1 1 1 --seed 42

## 3. Build Training Mixtures

We create four training mixtures, each containing 5,000 samples,
to compare different task proportions.

The order of `--ratios` is always **IF, Math, Coding**.

| Mixture | IF | Math | Coding | Total |
|---|---:|---:|---:|---:|
| Balanced | 1667 | 1667 | 1666 | 5000 |
| IF-heavy | 3000 | 1000 | 1000 | 5000 |
| Math-heavy | 1000 | 3000 | 1000 | 5000 |
| Code-heavy | 1000 | 1000 | 3000 | 5000 |

In [ ]:
# Balanced
!python build_mixture.py --size 5000 --ratios 1 1 1 --output data/mixtures/balanced_5k.jsonl --seed 42

# IF-heavy
!python build_mixture.py --size 5000 --ratios 3 1 1 --output data/mixtures/if_heavy_5k.jsonl --seed 42

# Math-heavy
!python build_mixture.py --size 5000 --ratios 1 3 1 --output data/mixtures/math_heavy_5k.jsonl --seed 42

# Code-heavy
!python build_mixture.py --size 5000 --ratios 1 1 3 --output data/mixtures/code_heavy_5k.jsonl --seed 42

## 4. Generated Files

```text
data/
├── full/
│   ├── all.jsonl
│   ├── if.jsonl
│   ├── math.jsonl
│   ├── coding.jsonl
│   └── manifest.json
├── validation.jsonl
├── validation.manifest.json
└── mixtures/
    ├── balanced_5k.jsonl
    ├── balanced_5k.manifest.json
    ├── if_heavy_5k.jsonl
    ├── if_heavy_5k.manifest.json
    ├── math_heavy_5k.jsonl
    ├── math_heavy_5k.manifest.json
    ├── code_heavy_5k.jsonl
    └── code_heavy_5k.manifest.json
```

- `full/`: All cleaned data, organized by task.
- `validation.jsonl`: The fixed validation dataset.
- `mixtures/`: Training datasets for individual experiments.
- `.manifest.json`: Metadata for reproducibility, including sample
  IDs, sources, categories, sampling ratios, and random seeds.

Validation and training JSONL files contain only `messages`.

## 5. Command-Line Arguments

### `prepare_data.py`

| Argument | Default | Description |
|---|---|---|
| `--output-dir` | `data` | Output directory for cleaned data. |
| `--revision` | Latest revision | Hugging Face dataset revision. |
| `--force` | False | Overwrite existing prepared data. |

Example:

```bash
python prepare_data.py --output-dir data --force
```

### `build_mixture.py`

| Argument | Default | Description |
|---|---|---|
| `--create-validation` | False | Create validation instead of training data. |
| `--size` | Required | Total number of samples. |
| `--ratios` | `1 1 1` | Relative proportions of IF, Math, and Coding. |
| `--seed` | `42` | Random seed for sampling. |
| `--source-dir` | `data/full` | Directory containing cleaned data. |
| `--output` | `data/validation.jsonl` for validation; required for training | Output JSONL path. |
| `--validation` | `data/validation.jsonl` | Validation dataset to exclude from training. |
| `--exclude-manifest` | None | Exclude sample IDs from another dataset manifest. Can be specified multiple times. |
| `--force` | False | Overwrite existing output files. |

### Additional Examples

Create a larger training mixture:

```bash
python build_mixture.py --size 30000 --ratios 1 2 1 --output data/mixtures/math_heavy_30k.jsonl
```

Overwrite an existing training mixture:

```bash
python build_mixture.py --size 5000 --ratios 1 1 1 --output data/mixtures/balanced_5k.jsonl --force
```

Replace the validation set:

```bash
python build_mixture.py --create-validation --size 900 --ratios 1 1 1 --seed 123 --force
```

Exclude samples from a previous training dataset:

```bash
python build_mixture.py --size 10000 --ratios 1 1 1 \
    --exclude-manifest data/mixtures/balanced_5k.manifest.json \
    --output data/mixtures/stage2_10k.jsonl
```

**Note:** Different training mixtures may share samples. This is
intentional for comparing task proportions. All training mixtures
must be disjoint from the validation set.

For the complete CLI documentation:

```bash
python prepare_data.py --help
python build_mixture.py --help
```